# 01. Data Audit — 테이블 인벤토리와 데이터 품질

> 이 노트북은 **설명용**이다. 계산 로직은 모두 `src/credit_risk/eda.py`, `src/credit_risk/stats.py`에 있으며, 보고서는 `python -m credit_risk.cli eda`로 재현한다.
>
> 용어: TARGET 평균 = **관측 상환곤란율 (Observed Difficulty Rate)**, AMT_CREDIT = **Exposure Proxy (AMT_CREDIT)**. 모든 관계는 연관이며 인과가 아니다. 합성 데이터가 적재된 경우 결과는 해석에 사용하지 않는다.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("DATA_DIR", "./data/raw")  # only used for the data directory; DB creds come from .env

import pandas as pd
from credit_risk.config import get_settings
from credit_risk.db import connect
from credit_risk import eda, stats

settings = get_settings()
pd.set_option("display.max_columns", 50)

## 데이터 출처 판별
`raw.columns_description` 설명이 `[synthetic]`으로 시작하면 합성 테스트 데이터다.

In [ ]:
with connect(settings) as conn:
    is_synth = eda.detect_synthetic(conn)
print('synthetic:', is_synth)
if is_synth:
    print(eda.SYNTHETIC_BANNER)

## 테이블 인벤토리
행·열·용량·키 중복·결측률을 **SQL 집계**로 계산한다 (대형 raw 테이블을 pandas로 읽지 않음).

In [ ]:
with connect(settings) as conn:
    inventory, column_nulls = eda.table_inventory(conn, settings)
    dq = eda.dq_status_summary(conn)
inventory

In [ ]:
dq  # audit.data_quality_result 최신 결과의 상태별 건수

## 컬럼별 결측률 상위

In [ ]:
column_nulls.sort_values('null_ratio', ascending=False).head(30)

## 결측 자체와 TARGET
집계 Feature(bur_/prev_/inst_/pos_/cc_)의 NULL은 '이력 없음'(structural)이므로 0과 구분한다.

In [ ]:
with connect(settings) as conn:
    cols = eda.table_columns(conn, 'analytics', 'customer_feature_mart')['column_name']
    skip = {'sk_id_curr', 'source', 'target', 'feature_version', 'built_at'}
    miss = eda.missingness_vs_target(conn, [c for c in cols if c not in skip])
miss.query('missing_ratio >= 0.01').sort_values('effect_size', ascending=False).head(20)